# GitHub Excel ile CP-SAT üretim planlama · v6.4

Colab'da **Çalışma zamanı → Tümünü çalıştır** seçin. Notebook bütün Python kodunu içerir; ayrı `.py` veya JSON dosyası gerektirmez.

Önce revize giriş Excel'ini GitHub'a `CP_SAT_input_data_v1.xlsx` adıyla yükleyin. Eski şema sessizce kabul edilmez. Çalıştırma hücresi her seferinde güncel Excel'i indirir.

Miktarlar **ton**, kapasiteler **ton/saat**, hazırlıklar **dakika**. Üretim adımı Excel'de kg olarak tanımlıdır. `HARD` hedefler tamdır; `SOFT` hedeflerde minimum yüzde zorunludur. Küçük `TargetPriority` daha önceliklidir. Amaç sırası ve süreler Excel'den gelir.


In [ ]:
%pip install -q ortools==9.15.6755 openpyxl==3.1.5


## Model ve Excel işlemleri
Bu hücre tek başına tüm uygulamayı içerir.


In [ ]:
#!/usr/bin/env python3
"""Excel -> OR-Tools CP-SAT -> Excel, input schema 6.4.

Install: pip install ortools==9.15.6755 openpyxl==3.1.5
GitHub:  python solve_v63_cp_sat.py
Local:   python solve_v63_cp_sat.py input.xlsx -o CP_SAT_output.xlsx

All business rules and solver budgets come from the input Excel. Smaller
TargetPriority numbers are served first within the MIN_UNMET objective.
Quantities in the workbook are tons; internal quantities are integer kg.
"""
from __future__ import annotations

import argparse
import hashlib
import io
import math
import re
import time
import zipfile
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
from decimal import Decimal, ROUND_CEILING, ROUND_FLOOR
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlsplit, urlunsplit
from urllib.request import Request, urlopen

import ortools
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from ortools.sat.python import cp_model

VERSION = "6.4.0"
SCHEMA_VERSION = "6.4"
INPUT_GITHUB_URL = (
    "https://github.com/hadokan/CPSAT-ProductionX/blob/main/"
    "CP_SAT_input_data_v1.xlsx"
)
WORK_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
INPUT_XLSX = WORK_DIR / "CP_SAT_input_data_v1.xlsx"
OUTPUT_XLSX = WORK_DIR / "CP_SAT_output.xlsx"

# These descriptions are also used to build the revised input workbook.
SETTING_ROWS = [
    ("SchemaVersion", "6.4", "Giriş şeması. Bu sürüm yalnızca 6.4 destekler."),
    ("QuantityUnit", "t", "Tüm _t alanları ton, Capacity_tph ton/saat, hazırlık süreleri dakika."),
    ("QuantityStep_kg", 1, "Üretim miktarının kg cinsinden adımı. Hedefler bu adımın katı olmalıdır."),
    ("DefaultPreparationMin", 40, "Preparation_min boş ise kullanılan hazırlık süresi, dakika."),
    ("AllowFinalRunBelowMinimum", 1, "1: yalnız hedefi tamamlayan son run için minimum miktar istisnası."),
    ("AllowSurplusProduction", 0, "Fazla üretim yasak. Bu sürümde yalnız 0 desteklenir."),
    ("PreventProductReturn", 1, "Makinede A-B-A yasak; boş/kapalı vardiya ürün durumunu sıfırlamaz. Yalnız 1 desteklenir."),
    ("Solver", "CP_SAT", "Kullanılan çözücü. Yalnız CP_SAT desteklenir."),
    ("TotalTimeLimitSec", 240, "Tüm CP-SAT aşamaları için toplam çözüm süresi bütçesi, saniye; Excel I/O hariç."),
    ("StageTimeLimitSec", 45, "objectives.TimeLimit_sec boşsa her gerçek çözüm aşamasının süre sınırı, saniye."),
    ("NumSearchWorkers", 1, "CP-SAT çalışan sayısı. Tekrarlanabilir karşılaştırma için 1 önerilir."),
    ("RandomSeed", 20261008, "Rastgelelik tohumu. Aynı sürüm/ortam ve 1 çalışan ile karşılaştırma yapılabilir."),
    ("LogSearchProgress", 0, "1: ayrıntılı CP-SAT arama günlüğünü konsola yaz."),
    ("StageCompletionPolicy", "ACCEPT_FEASIBLE", "ACCEPT_FEASIBLE: süre dolunca bulunan en iyiyi sabitle ve kanıtlanmadığını raporla. REQUIRE_OPTIMAL: kanıt gelmezse alt aşamalara geçme."),
    ("InputValidation", "STRICT", "Şema, kimlik, referans, sayı ve kural çelişkilerini kontrol et. Yalnız STRICT desteklenir."),
    ("UnknownParameterPolicy", "ERROR", "Desteklenmeyen alan, ayar, amaç veya kapsam açık hata verir. Yalnız ERROR desteklenir."),
    ("TargetWarningCapacityFactor", 5, "Hedef iyimser ürün kapasitesinin bu katını aşarsa birim/ölçek uyarısı ver; hedefi değiştirme."),
]

HEADERS = {
    "products": ["Product_ID", "Product", "Family", "Salt_Status", "Sequence_Order"],
    "machines": ["Machine_ID", "Machine", "Active", "Priority"],
    "machine_product": ["Machine_ID", "Product_ID", "Allowed", "MachinePriority", "Capacity_tph", "MinRun_t", "MaxDaily_t", "Preparation_min"],
    "targets": ["Product_ID", "Product", "Target_t", "TargetPriority", "TargetMode", "MinRequired_pct", "MinRequired_t"],
    "calendar": ["Machine_ID", "Day", "Shift", "Available", "Hours", "Reason"],
    "sequence_rules": ["Rule_ID", "Family", "Before_Product_ID", "After_Product_ID", "Scope", "Hard"],
    "changeover_rules": ["Rule_ID", "Scope", "Max_Products_Per_Shift", "Max_Changeovers_Per_Day", "Hard", "Description"],
    "allocation_rules": ["Rule_ID", "Scope", "Prefer_Single_Machine", "Preferred_Max_Machines", "Allow_Split_If_Required", "Hard", "Description", "Objective_ID"],
    "initial_state": ["Machine_ID", "Current_Product_ID", "Run_Produced_t", "Setup_Completed"],
    "objectives": ["Priority", "Objective_ID", "Direction", "Enabled", "Description", "TimeLimit_sec"],
    "settings": ["Parameter", "Value", "Description"],
}
OBJECTIVES = {
    "MIN_CHANGEOVERS": "MIN", "MIN_UNMET": "MIN", "MAX_PRODUCTION": "MAX",
    "MIN_TOTAL_PREPARATION": "MIN", "MIN_PRODUCT_MACHINE_SPLITS": "MIN",
    "MIN_FAMILY_MACHINE_SPLITS": "MIN", "MIN_MACHINE_PRIORITY_COST": "MIN",
}


class InputValidationError(ValueError):
    pass


def fail(message):
    raise InputValidationError(message)


def text(value):
    return "" if value is None else str(value).strip()


def number(value, label, minimum=None, maximum=None):
    if isinstance(value, bool) or not isinstance(value, (int, float, Decimal)):
        fail(f"{label}: sayısal Excel hücresi gerekli; bulunan değer: {value!r}.")
    n = Decimal(str(value))
    if not n.is_finite() or (minimum is not None and n < Decimal(str(minimum))) or (maximum is not None and n > Decimal(str(maximum))):
        fail(f"{label}: geçersiz sayı/aralık: {value!r}.")
    return n


def integer(value, label, minimum=0, maximum=None):
    n = number(value, label, minimum, maximum)
    if n != int(n):
        fail(f"{label}: tam sayı gerekli.")
    return int(n)


def flag(value, label):
    return integer(value, label, 0, 1)


def tonnes_to_kg(value, label, step=1, exact=False):
    n = number(value, label, 0) * 1000
    nearest = round(n / step) * step
    if exact and abs(n - nearest) > Decimal("0.00001"):
        fail(f"{label}: hedef {step} kg adımının katı olmalı; giriş birimi tondur.")
    return int(nearest) if exact else int((n / step).to_integral_value(rounding=ROUND_CEILING)) * step


def floor_capacity(rate, hours, setup_min, step):
    net = max(Decimal(0), Decimal(str(hours)) - Decimal(str(setup_min)) / 60)
    value = Decimal(str(rate)) * net * 1000
    return int((value / step + Decimal("0.00000001")).to_integral_value(rounding=ROUND_FLOOR)) * step


def github_to_raw_url(url):
    parts = urlsplit(url.strip())
    if parts.scheme != "https":
        raise ValueError("GitHub bağlantısı https:// ile başlamalı.")
    if parts.hostname == "github.com":
        path = parts.path.strip("/").split("/")
        if len(path) < 5 or path[2] != "blob":
            raise ValueError("github.com/kullanıcı/depo/blob/dal/dosya.xlsx biçimini kullanın.")
        return urlunsplit(("https", "raw.githubusercontent.com", "/" + "/".join(path[:2] + path[3:]), "", ""))
    if parts.hostname != "raw.githubusercontent.com":
        raise ValueError("Yalnız github.com/blob veya raw.githubusercontent.com bağlantıları desteklenir.")
    return urlunsplit((parts.scheme, parts.netloc, parts.path, parts.query, ""))


def download_input_excel(url, destination):
    """Always download; never fall back silently to an old local workbook."""
    destination = Path(destination)
    raw = github_to_raw_url(url)
    request = Request(raw, headers={"User-Agent": "CP-SAT-Production-Planner", "Cache-Control": "no-cache"})
    try:
        with urlopen(request, timeout=45) as response:
            content = response.read(50 * 1024 * 1024 + 1)
    except (HTTPError, URLError, TimeoutError) as exc:
        raise RuntimeError(f"GitHub Excel indirilemedi. Dosya adını, dalı ve herkese açık erişimi kontrol edin. {exc}") from exc
    if len(content) > 50 * 1024 * 1024:
        raise RuntimeError("İndirilen giriş dosyası 50 MB sınırını aşıyor.")
    try:
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            if not {"[Content_Types].xml", "xl/workbook.xml"}.issubset(z.namelist()):
                raise ValueError("XLSX yapısı bulunamadı")
    except (zipfile.BadZipFile, ValueError) as exc:
        raise RuntimeError("İndirilen içerik geçerli XLSX değil; GitHub dosya adresini kontrol edin.") from exc
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".download")
    temporary.write_bytes(content)
    temporary.replace(destination)
    print(f"Giriş Excel indirildi: {destination}")
    return destination


def read_tables(path):
    """Read constants, not stale formula caches. MinRequired_t is derived in Python."""
    try:
        workbook = load_workbook(path, read_only=True, data_only=False)
    except Exception as exc:
        fail(f"Giriş Excel açılamadı: {exc}")
    tables = {}
    try:
        unknown = set(workbook.sheetnames) - set(HEADERS)
        missing = set(HEADERS) - set(workbook.sheetnames)
        if unknown or missing:
            fail(f"Sayfa şeması hatası. Eksik: {sorted(missing)}; desteklenmeyen: {sorted(unknown)}.")
        for sheet in workbook.worksheets:
            rows = list(sheet.iter_rows(values_only=True))
            if not rows:
                fail(f"{sheet.title}: başlık satırı eksik.")
            headers = [text(v) for v in rows[0]]
            while headers and not headers[-1]:
                headers.pop()
            if len(headers) != len(set(headers)) or set(headers) != set(HEADERS[sheet.title]):
                fail(f"{sheet.title}: sütunlar tam olarak şu alanları içermeli: {HEADERS[sheet.title]}.")
            parsed = []
            for rowno, row in enumerate(rows[1:], 2):
                if not any(v is not None and text(v) for v in row):
                    continue
                if any(v is not None for v in row[len(headers):]):
                    fail(f"{sheet.title}!{rowno}: başlıksız sütunda veri var.")
                item = dict(zip(headers, row[:len(headers)]))
                for name, value in list(item.items()):
                    if not (isinstance(value,str) and value.startswith("=")):
                        continue
                    if sheet.title=="targets" and name=="MinRequired_t":
                        continue  # derived again from the authoritative target and percentage
                    if sheet.title=="machine_product" and name in ("MinRun_t","MaxDaily_t"):
                        # Preserve the template's =Erow*3 / =Erow*8 formulas and
                        # recalculate them from Capacity_tph, never an Excel cache.
                        match=re.fullmatch(rf"=\s*\$?E\$?{rowno}\s*\*\s*([0-9]+(?:\.[0-9]+)?)\s*",value,re.I)
                        if match:
                            rate=number(item["Capacity_tph"],f"machine_product!{rowno}.Capacity_tph",0)
                            item[name]=float(rate*Decimal(match.group(1)))
                            continue
                    fail(f"{sheet.title}!{rowno}.{name}: desteklenmeyen formül; sabit değer veya şablondaki standart formülü kullanın.")
                item["_row"] = rowno
                parsed.append(item)
            tables[sheet.title] = parsed
    finally:
        workbook.close()
    return tables


@dataclass
class Data:
    settings: dict
    products: list
    machines: list
    targets: dict
    slots: list
    hours: dict
    reasons: dict
    machine_product: dict
    initial: dict
    sequences: list
    max_changes: int
    allocation: dict
    objectives: list
    warnings: list = field(default_factory=list)


def unique(rows, keys, sheet):
    seen = set()
    for r in rows:
        key = tuple(text(r.get(k)) for k in keys)
        if any(not v for v in key) or key in seen:
            fail(f"{sheet}!{r.get('_row', '?')}: boş veya mükerrer anahtar {key}.")
        seen.add(key)


def parse_data(tables):
    srows = tables["settings"]
    unique(srows, ["Parameter"], "settings")
    settings = {text(r["Parameter"]): r["Value"] for r in srows}
    supported = {r[0] for r in SETTING_ROWS}
    if set(settings) != supported:
        fail(f"settings: eksik ayarlar {sorted(supported-set(settings))}; desteklenmeyen ayarlar {sorted(set(settings)-supported)}.")
    for name, expected in {"SchemaVersion":"6.4", "QuantityUnit":"t", "Solver":"CP_SAT", "InputValidation":"STRICT", "UnknownParameterPolicy":"ERROR"}.items():
        if text(settings[name]) != expected:
            fail(f"settings.{name}: bu sürümde yalnız {expected!r} desteklenir; {settings[name]!r} bulundu.")
    for name, expected in {"PreventProductReturn":1, "AllowSurplusProduction":0}.items():
        if flag(settings[name], f"settings.{name}") != expected:
            fail(f"settings.{name}: bu sürümde yalnız {expected} desteklenir.")
    for name in ("AllowFinalRunBelowMinimum", "LogSearchProgress"):
        settings[name] = flag(settings[name], f"settings.{name}")
    for name, lower in (("QuantityStep_kg",1), ("NumSearchWorkers",1), ("RandomSeed",0)):
        settings[name] = integer(settings[name], f"settings.{name}", lower, 2147483647)
    for name, lower in (("DefaultPreparationMin",0), ("TotalTimeLimitSec",0.01), ("StageTimeLimitSec",0.01), ("TargetWarningCapacityFactor",1)):
        settings[name] = float(number(settings[name], f"settings.{name}", lower))
    if text(settings["StageCompletionPolicy"]) not in ("ACCEPT_FEASIBLE", "REQUIRE_OPTIMAL"):
        fail("settings.StageCompletionPolicy: ACCEPT_FEASIBLE veya REQUIRE_OPTIMAL olmalı.")
    step = settings["QuantityStep_kg"]
    products = tables["products"]
    machines_all = tables["machines"]
    unique(products, ["Product_ID"], "products")
    unique(machines_all, ["Machine_ID"], "machines")
    unique(machines_all, ["Machine"], "machines")
    if not products or not machines_all:
        fail("En az bir ürün ve bir makine gerekli.")
    pids = {text(r["Product_ID"]) for r in products}
    mids = {text(r["Machine_ID"]) for r in machines_all}
    pmap = {}
    for r in products:
        r["Product_ID"] = text(r["Product_ID"])
        for fieldname in ("Product", "Family"):
            r[fieldname] = text(r[fieldname])
            if not r[fieldname]: fail(f"products.{r['Product_ID']}.{fieldname}: boş olamaz.")
        if text(r["Salt_Status"]) not in ("UNSALTED", "SALTED", "NONE"):
            fail(f"products.{r['Product_ID']}.Salt_Status: geçersiz değer.")
        r["Sequence_Order"] = integer(r["Sequence_Order"], "products.Sequence_Order", 0)
        pmap[r["Product_ID"]] = r
    for r in machines_all:
        r["Machine_ID"] = text(r["Machine_ID"])
        r["Active"] = flag(r["Active"], "machines.Active")
        r["Priority"] = integer(r["Priority"], "machines.Priority", 1)
    machines = [r for r in machines_all if r["Active"]]
    if not machines:
        fail("Aktif makine yok.")
    active = {r["Machine_ID"] for r in machines}
    unique(tables["targets"], ["Product_ID"], "targets")
    if {text(r["Product_ID"]) for r in tables["targets"]} != pids:
        fail("targets: her ürün için tam bir hedef satırı gerekli; bilinmeyen ürün olamaz.")
    targets = {}
    for r in tables["targets"]:
        p = text(r["Product_ID"])
        if text(r["Product"]) != pmap[p]["Product"]:
            fail(f"targets.{p}.Product: ürün adı products sayfasıyla uyuşmuyor.")
        target = tonnes_to_kg(r["Target_t"], f"targets.{p}.Target_t", step, exact=True)
        mode = text(r["TargetMode"])
        if mode not in ("HARD", "SOFT"): fail(f"targets.{p}.TargetMode: HARD veya SOFT olmalı.")
        pct = number(r["MinRequired_pct"], f"targets.{p}.MinRequired_pct", 0, 1)
        minimum = target if mode == "HARD" else int((Decimal(target) * pct / step).to_integral_value(rounding=ROUND_CEILING)) * step
        displayed = r["MinRequired_t"]
        if displayed is not None and not (isinstance(displayed,str) and displayed.startswith("=")):
            if abs(number(displayed, f"targets.{p}.MinRequired_t",0)*1000-minimum) > Decimal("0.00001"):
                fail(f"targets.{p}.MinRequired_t: hesaplanan {minimum/1000:g} ton ile uyuşmuyor; alan otomatik hesaplanmalı.")
        targets[p] = {"kg":target, "min_kg":minimum, "pct":float(pct), "mode":mode,
                      "priority":integer(r["TargetPriority"], f"targets.{p}.TargetPriority",1)}
    unique(tables["calendar"], ["Machine_ID","Day","Shift"], "calendar")
    hours, reasons, pairs = {}, {}, set()
    for r in tables["calendar"]:
        m = text(r["Machine_ID"])
        if m not in mids: fail(f"calendar: bilinmeyen makine {m}.")
        match = re.fullmatch(r"Day-([1-9]\d*)", text(r["Day"]))
        if not match: fail(f"calendar.Day: Day-1 gibi pozitif gün etiketi gerekli; {r['Day']!r} bulundu.")
        slot = (int(match.group(1)), integer(r["Shift"],"calendar.Shift",1))
        available = flag(r["Available"],"calendar.Available")
        h = float(number(r["Hours"],"calendar.Hours",0,24))
        if bool(available) != (h>0): fail(f"calendar.{m}.{r['Day']}.{slot[1]}: Available ve Hours çelişiyor.")
        if m in active:
            pairs.add(slot); hours[m,slot] = h; reasons[m,slot] = text(r["Reason"])
    if not pairs: fail("calendar: aktif makineler için takvim gerekli.")
    slots = sorted(pairs)
    for m in active:
        for slot in slots:
            if (m,slot) not in hours: fail(f"calendar: {m}, Day-{slot[0]}, Shift-{slot[1]} satırı eksik.")
    unique(tables["machine_product"],["Machine_ID","Product_ID"],"machine_product")
    mp, warnings = {}, []
    for r in tables["machine_product"]:
        m,p = text(r["Machine_ID"]),text(r["Product_ID"])
        if m not in mids or p not in pids: fail(f"machine_product: bilinmeyen referans {m}/{p}.")
        allowed = flag(r["Allowed"],"machine_product.Allowed")
        rate = float(number(r["Capacity_tph"],f"machine_product.{m}.{p}.Capacity_tph",0))
        if not allowed: continue
        if rate <= 0: fail(f"machine_product.{m}.{p}: Allowed=1 için Capacity_tph pozitif olmalı.")
        minimum = tonnes_to_kg(r["MinRun_t"],f"machine_product.{m}.{p}.MinRun_t",step)
        daily_decimal = number(r["MaxDaily_t"],f"machine_product.{m}.{p}.MaxDaily_t",0)*1000
        daily = int((daily_decimal/step + Decimal('0.00000001')).to_integral_value(rounding=ROUND_FLOOR))*step
        preparation = r["Preparation_min"]
        if preparation is None or text(preparation)=="":
            preparation = settings["DefaultPreparationMin"]
            warnings.append(f"{m}/{p}: boş Preparation_min için {preparation:g} dakika varsayılanı kullanıldı.")
        preparation = float(number(preparation,f"machine_product.{m}.{p}.Preparation_min",0,1440))
        if abs(preparation*1000-round(preparation*1000))>1e-6:
            fail(f"machine_product.{m}.{p}.Preparation_min: en fazla 3 ondalık basamak desteklenir.")
        pr = integer(r["MachinePriority"],f"machine_product.{m}.{p}.MachinePriority",1)
        if m in active: mp[m,p] = {"rate":rate,"min_kg":minimum,"daily_kg":daily,"prep":preparation,"priority":pr}
    unique(tables["initial_state"],["Machine_ID"],"initial_state")
    if {text(r["Machine_ID"]) for r in tables["initial_state"]} != mids:
        fail("initial_state: her makine için tam bir satır gerekli.")
    initial = {}
    for r in tables["initial_state"]:
        m,p = text(r["Machine_ID"]),text(r["Current_Product_ID"])
        if p and p not in pids: fail(f"initial_state.{m}: bilinmeyen ürün {p}.")
        qty = tonnes_to_kg(r["Run_Produced_t"],f"initial_state.{m}.Run_Produced_t",step,exact=True)
        prepared = flag(r["Setup_Completed"],f"initial_state.{m}.Setup_Completed")
        if not p and (qty or prepared): fail(f"initial_state.{m}: ürün boşken miktar/hazırlık tamamlanmış olamaz.")
        if m in active: initial[m] = {"product":p,"kg":qty,"prepared":prepared}
    rules = tables["changeover_rules"]
    if len(rules)!=1: fail("changeover_rules: bu sürümde tek MACHINE_DAY kuralı gerekli.")
    r = rules[0]
    if text(r["Scope"])!="MACHINE_DAY" or flag(r["Hard"],"changeover_rules.Hard")!=1 or integer(r["Max_Products_Per_Shift"],"changeover_rules.Max_Products_Per_Shift",1)!=1:
        fail("changeover_rules: yalnız Hard=1, Scope=MACHINE_DAY, Max_Products_Per_Shift=1 desteklenir.")
    max_changes = integer(r["Max_Changeovers_Per_Day"],"changeover_rules.Max_Changeovers_Per_Day",0)
    unique(tables["sequence_rules"],["Rule_ID"],"sequence_rules")
    unique(tables["sequence_rules"],["Before_Product_ID","After_Product_ID"],"sequence_rules")
    sequences, graph = [], defaultdict(list)
    for r in tables["sequence_rules"]:
        a,b = text(r["Before_Product_ID"]),text(r["After_Product_ID"])
        if a not in pids or b not in pids or a==b: fail(f"sequence_rules: geçersiz ürün çifti {a}/{b}.")
        if text(r["Scope"])!="SAME_MACHINE" or flag(r["Hard"],"sequence_rules.Hard")!=1:
            fail("sequence_rules: yalnız Scope=SAME_MACHINE ve Hard=1 desteklenir.")
        if text(r["Family"]) and (pmap[a]["Family"]!=text(r["Family"]) or pmap[b]["Family"]!=text(r["Family"])):
            fail(f"sequence_rules: {a}/{b} Family alanı ile çelişiyor.")
        if pmap[a]["Family"]==pmap[b]["Family"] and pmap[a]["Sequence_Order"]>=pmap[b]["Sequence_Order"]:
            fail(f"sequence_rules: {a}/{b}, products.Sequence_Order ile çelişiyor.")
        sequences.append((a,b)); graph[a].append(b)
    visiting, visited = set(),set()
    def visit(p):
        if p in visiting: fail("sequence_rules: döngüsel/çelişkili ürün sırası.")
        if p in visited: return
        visiting.add(p)
        for q in graph[p]: visit(q)
        visiting.remove(p); visited.add(p)
    for p in pids: visit(p)
    unique(tables["allocation_rules"],["Rule_ID"],"allocation_rules")
    unique(tables["allocation_rules"],["Scope"],"allocation_rules")
    allocation = {}
    for r in tables["allocation_rules"]:
        scope = text(r["Scope"])
        expected = {"ALL_PRODUCTS":"MIN_PRODUCT_MACHINE_SPLITS","ALL_FAMILIES":"MIN_FAMILY_MACHINE_SPLITS"}.get(scope)
        if not expected or text(r["Objective_ID"])!=expected:
            fail(f"allocation_rules: desteklenmeyen kapsam/amaç: {scope}/{r['Objective_ID']}.")
        allocation[scope] = {
            "preferred":flag(r["Prefer_Single_Machine"],"allocation_rules.Prefer_Single_Machine"),
            "max":integer(r["Preferred_Max_Machines"],"allocation_rules.Preferred_Max_Machines",1),
            "allow_split":flag(r["Allow_Split_If_Required"],"allocation_rules.Allow_Split_If_Required"),
            "hard":flag(r["Hard"],"allocation_rules.Hard"),
        }
    unique(tables["objectives"],["Objective_ID"],"objectives")
    unique(tables["objectives"],["Priority"],"objectives")
    objectives = []
    for r in tables["objectives"]:
        name = text(r["Objective_ID"])
        if name not in OBJECTIVES or text(r["Direction"])!=OBJECTIVES[name]:
            fail(f"objectives: desteklenmeyen amaç/yön: {name}/{r['Direction']}.")
        enabled = flag(r["Enabled"],"objectives.Enabled")
        priority = integer(r["Priority"],"objectives.Priority",1)
        seconds = settings["StageTimeLimitSec"] if r["TimeLimit_sec"] is None else float(number(r["TimeLimit_sec"],"objectives.TimeLimit_sec",0.01))
        if enabled:
            if name in ("MIN_PRODUCT_MACHINE_SPLITS","MIN_FAMILY_MACHINE_SPLITS"):
                scope = "ALL_PRODUCTS" if name=="MIN_PRODUCT_MACHINE_SPLITS" else "ALL_FAMILIES"
                if scope not in allocation: fail(f"objectives.{name}: allocation_rules.{scope} gerekli.")
            objectives.append({"name":name,"priority":priority,"seconds":seconds,"direction":OBJECTIVES[name]})
    if not objectives: fail("objectives: en az bir amaç Enabled=1 olmalı.")
    objectives.sort(key=lambda r:r["priority"])
    return Data(settings,products,machines,targets,slots,hours,reasons,mp,initial,sequences,max_changes,allocation,objectives,warnings)


def load_data(path):
    return parse_data(read_tables(path))


def optimistic_capacity(d,p):
    step = d.settings["QuantityStep_kg"]
    total = 0
    for machine in d.machines:
        m = machine["Machine_ID"]
        if (m,p) not in d.machine_product: continue
        a = d.machine_product[m,p]
        for day in sorted({s[0] for s in d.slots}):
            total += min(a["daily_kg"],sum(floor_capacity(a["rate"],d.hours[m,s],0,step) for s in d.slots if s[0]==day))
    return total


def preflight(d):
    issues = []
    for p,r in d.targets.items():
        cap = optimistic_capacity(d,p)
        if cap and r["kg"] > cap*d.settings["TargetWarningCapacityFactor"]:
            d.warnings.append(f"{p}: {r['kg']/1000:g} ton hedef kapasiteye göre şüpheli; ton/kg ölçeğini kontrol edin. Hedef değiştirilmedi.")
        if r["min_kg"]>cap:
            issues.append(f"{p}: zorunlu minimum {r['min_kg']/1000:g} ton; hazırlıklar hariç iyimser kapasite en fazla {cap/1000:g} ton.")
    return issues


def build_model(d):
    model = cp_model.CpModel()
    mids = [r["Machine_ID"] for r in d.machines]
    pids = [r["Product_ID"] for r in d.products]
    step = d.settings["QuantityStep_kg"]
    T = len(d.slots)
    x, q, starts, y, amount, pos, last, setup, change, first = [{} for _ in range(10)]
    for (m,p), a in d.machine_product.items():
        limit = d.targets[p]["kg"]//step
        y[m,p] = model.NewBoolVar(f"use_{m}_{p}")
        amount[m,p] = model.NewIntVar(0,limit,f"amount_{m}_{p}")
        pos[m,p] = model.NewIntVar(0,T-1,f"firstpos_{m}_{p}")
        last[m,p] = model.NewIntVar(0,T-1,f"lastpos_{m}_{p}")
        seen = 0
        for t,s in enumerate(d.slots):
            if d.hours[m,s]<=0: continue
            key = (m,p,t)
            x[key] = model.NewBoolVar(f"x_{m}_{p}_{t}")
            cap = floor_capacity(a["rate"],d.hours[m,s],0,step)//step
            cap_prepared = floor_capacity(a["rate"],d.hours[m,s],a["prep"],step)//step
            q[key] = model.NewIntVar(0,min(cap,limit),f"q_{m}_{p}_{t}")
            starts[key] = model.NewBoolVar(f"start_{m}_{p}_{t}")
            model.Add(q[key]<=min(cap,limit)*x[key])
            model.Add(q[key]>=x[key])
            model.Add(starts[key]>=x[key]-seen)
            model.Add(starts[key]<=x[key])
            model.Add(starts[key]<=1-seen)
            next_seen = model.NewBoolVar(f"seen_{m}_{p}_{t}")
            model.AddMaxEquality(next_seen,[seen,x[key]])
            seen = next_seen
            setup[key] = model.NewBoolVar(f"setup_{m}_{p}_{t}")
            # The initial product cannot return after another product. Therefore,
            # if assigned, this is a continuation even after idle/closed shifts.
            skip = d.initial[m]["product"]==p and d.initial[m]["prepared"]
            model.Add(setup[key]==(0 if skip else starts[key]))
            model.Add(q[key]<=cap*x[key]-(cap-cap_prepared)*setup[key])
        keys = [(m,p,t) for t in range(T) if (m,p,t) in x]
        model.Add(sum(starts[k] for k in keys)==y[m,p])
        model.Add(amount[m,p]==sum(q[k] for k in keys))
        model.Add(pos[m,p]==sum(k[2]*starts[k] for k in keys))
        model.AddMaxEquality(last[m,p],[k[2]*x[k] for k in keys] or [0])
        for day in sorted({s[0] for s in d.slots}):
            model.Add(sum(q[k] for k in keys if d.slots[k[2]][0]==day)<=a["daily_kg"]//step)
    for m in mids:
        for t in range(T):
            model.Add(sum(x[m,p,t] for p in pids if (m,p,t) in x)<=1)
        ep = [p for p in pids if (m,p) in y]
        for i,a in enumerate(ep):
            for b in ep[i+1:]:
                order = model.NewBoolVar(f"order_{m}_{a}_{b}")
                model.Add(last[m,a]+1<=pos[m,b]).OnlyEnforceIf([y[m,a],y[m,b],order])
                model.Add(last[m,b]+1<=pos[m,a]).OnlyEnforceIf([y[m,a],y[m,b],order.Not()])
        used = model.NewBoolVar(f"machine_used_{m}")
        model.AddMaxEquality(used,[y[m,p] for p in ep] or [0])
        for p in ep:
            first[m,p] = model.NewBoolVar(f"first_{m}_{p}")
            model.Add(first[m,p]<=y[m,p])
            for other in ep:
                if other!=p:
                    model.Add(pos[m,p]<=pos[m,other]).OnlyEnforceIf([first[m,p],y[m,other]])
            if d.initial[m]["product"]==p:
                model.Add(first[m,p]==y[m,p])  # include initial-state no-return history
        model.Add(sum(first[m,p] for p in ep)==used)
        for p in ep:
            for t in range(T):
                key = (m,p,t)
                if key not in starts: continue
                change[key] = model.NewBoolVar(f"change_{m}_{p}_{t}")
                initial = d.initial[m]["product"]
                if not initial:
                    model.Add(change[key]>=starts[key]-first[m,p])
                    model.Add(change[key]<=starts[key])
                    model.Add(change[key]<=1-first[m,p])
                else:
                    model.Add(change[key]==(0 if initial==p else starts[key]))
        for day in sorted({s[0] for s in d.slots}):
            model.Add(sum(v for (mm,p,t),v in change.items() if mm==m and d.slots[t][0]==day)<=d.max_changes)
        for a,b in d.sequences:
            if (m,a) in y and (m,b) in y:
                model.Add(last[m,a]+1<=pos[m,b]).OnlyEnforceIf([y[m,a],y[m,b]])
            if d.initial[m]["product"]==b and (m,a) in y:
                model.Add(y[m,a]==0)
    total = {}
    for p in pids:
        target = d.targets[p]
        total[p] = model.NewIntVar(target["min_kg"]//step,target["kg"]//step,f"total_{p}")
        model.Add(total[p]==sum(amount[m,p] for m in mids if (m,p) in amount))
    exceptions = {}
    for (m,p),a in d.machine_product.items():
        credit = d.initial[m]["kg"] if d.initial[m]["product"]==p else 0
        minimum = max(0,a["min_kg"]-credit)//step
        if not d.settings["AllowFinalRunBelowMinimum"] or minimum==0:
            model.Add(amount[m,p]>=minimum*y[m,p])
            continue
        ex = model.NewBoolVar(f"final_below_min_{m}_{p}")
        exceptions[m,p] = ex
        model.Add(ex<=y[m,p])
        model.Add(amount[m,p]>=minimum*y[m,p]-minimum*ex)
        model.Add(amount[m,p]<=minimum-1).OnlyEnforceIf(ex)
        model.Add(total[p]==d.targets[p]["kg"]//step).OnlyEnforceIf(ex)
        for other in mids:
            if other!=m and (other,p) in y:
                model.Add(last[other,p]<=last[m,p]).OnlyEnforceIf([ex,y[other,p]])
    for p in pids:
        model.Add(sum(v for (m,pp),v in exceptions.items() if pp==p)<=1)
    families = list(dict.fromkeys(r["Family"] for r in d.products))
    family_of = {r["Product_ID"]:r["Family"] for r in d.products}
    family_use = {}
    for m in mids:
        for f in families:
            family_use[m,f] = model.NewBoolVar(f"family_{m}_{f}")
            model.AddMaxEquality(family_use[m,f],[y[m,p] for p in pids if (m,p) in y and family_of[p]==f] or [0])
    split_cost = {}
    for scope,groups,counts in (
        ("ALL_PRODUCTS",pids,{p:sum(y[m,p] for m in mids if (m,p) in y) for p in pids}),
        ("ALL_FAMILIES",families,{f:sum(family_use[m,f] for m in mids) for f in families}),
    ):
        terms = []
        rule = d.allocation.get(scope)
        if rule:
            for group in groups:
                count = counts[group]
                if rule["hard"] or not rule["allow_split"]:
                    model.Add(count<=rule["max"])
                if rule["preferred"]:
                    excess = model.NewIntVar(0,len(mids),f"excess_{scope}_{group}")
                    model.AddMaxEquality(excess,[0,count-rule["max"]])
                    terms.append(excess)
        split_cost[scope] = sum(terms)
    machine_rank = {r["Machine_ID"]:r["Priority"] for r in d.machines}
    multiplier = max(machine_rank.values())+1
    objectives = {
        "MIN_CHANGEOVERS": (sum(change.values()),1,"count"),
        "MIN_TOTAL_PREPARATION": (sum(round(d.machine_product[m,p]["prep"]*1000)*v for (m,p,t),v in setup.items()),0.001,"min"),
        "MIN_PRODUCT_MACHINE_SPLITS": (split_cost["ALL_PRODUCTS"],1,"count"),
        "MIN_FAMILY_MACHINE_SPLITS": (split_cost["ALL_FAMILIES"],1,"count"),
        "MIN_MACHINE_PRIORITY_COST": (sum(amount[m,p]*((a["priority"]-1)*multiplier+machine_rank[m]-1) for (m,p),a in d.machine_product.items()),step,"kg_rank"),
    }
    stages = []
    for item in d.objectives:
        name = item["name"]
        if name in ("MIN_UNMET","MAX_PRODUCTION"):
            priorities = sorted({r["priority"] for r in d.targets.values() if r["kg"]>0}) or [1]
            for rank in priorities:
                ps = [p for p in pids if d.targets[p]["priority"]==rank]
                expr = sum((d.targets[p]["kg"]//step-total[p]) if name=="MIN_UNMET" else total[p] for p in ps)
                stages.append({**item,"name":f"{name}_PRIORITY_{rank}","expr":expr,"factor":step,"unit":"kg"})
        else:
            expr,factor,unit = objectives[name]
            stages.append({**item,"expr":expr,"factor":factor,"unit":unit})
    variables = {"x":x,"q":q,"starts":starts,"y":y,"total":total,"setup":setup,"change":change,"first":first,"last":last}
    return model,variables,stages


def extract_plan(d,solver,variables):
    records = []
    step = d.settings["QuantityStep_kg"]
    q = variables["q"]
    for machine in d.machines:
        m = machine["Machine_ID"]
        previous = d.initial[m]["product"]
        prepared = d.initial[m]["prepared"]
        for t,s in enumerate(d.slots):
            selected = [(p,solver.Value(var)*step) for (mm,p,tt),var in q.items() if mm==m and tt==t and solver.Value(var)>0]
            if selected:
                p,kg = selected[0]
                prep = d.machine_product[m,p]["prep"] if previous!=p or not prepared else 0.0
                changed = int(bool(previous) and previous!=p)
                records.append({"m":m,"t":t,"p":p,"kg":kg,"Prep_min":prep,"Changeover":changed})
                previous,prepared = p,1
            else:
                records.append({"m":m,"t":t,"p":"","kg":0,"Prep_min":None,"Changeover":0})
    return records


def validate_plan(d,records):
    """Recompute every constraint from exported decisions, not solver variables."""
    errors = []
    mids = [r["Machine_ID"] for r in d.machines]
    step = d.settings["QuantityStep_kg"]
    by_slot = {}
    for r in records:
        key = (r["m"],r["t"])
        if key in by_slot: errors.append(f"Mükerrer plan satırı: {key}.")
        by_slot[key] = r
    expected = {(m,t) for m in mids for t in range(len(d.slots))}
    if set(by_slot)!=expected:
        return ["Plan satırları makine/takvim anahtarlarıyla uyuşmuyor."]
    totals, daily, changes, used, blocks = defaultdict(int),defaultdict(int),defaultdict(int),defaultdict(set),{}
    family_of = {r["Product_ID"]:r["Family"] for r in d.products}
    for m in mids:
        initial = d.initial[m]
        previous,prepared = initial["product"],initial["prepared"]
        history = [previous] if previous else []
        for t,s in enumerate(d.slots):
            r = by_slot[m,t]
            p,kg = r["p"],r["kg"]
            if not p:
                if kg or r["Prep_min"] not in (None,0): errors.append(f"{m}/{s}: boş vardiyada miktar/hazırlık.")
                continue
            if (m,p) not in d.machine_product:
                errors.append(f"{m}/{s}: yasak makine/ürün {p}."); continue
            a = d.machine_product[m,p]
            if not isinstance(kg,int) or kg<=0 or kg%step:
                errors.append(f"{m}/{s}: miktar pozitif ve {step} kg katı olmalı.")
            if d.hours[m,s]<=0: errors.append(f"{m}/{s}: kapalı vardiyada üretim.")
            required_prep = a["prep"] if previous!=p or not prepared else 0.0
            if r["Prep_min"] is None or abs(r["Prep_min"]-required_prep)>1e-7:
                errors.append(f"{m}/{s}: hazırlık {r['Prep_min']}; beklenen {required_prep} dakika.")
            occupied = kg/1000/a["rate"]+required_prep/60
            if occupied>d.hours[m,s]+1e-8:
                errors.append(f"{m}/{s}: hazırlık dahil {occupied:.9f} saat, kapasite {d.hours[m,s]:g} saat.")
            changed = int(bool(previous) and previous!=p)
            if r["Changeover"]!=changed: errors.append(f"{m}/{s}: yanlış değişim sayacı.")
            changes[m,s[0]] += changed
            if previous!=p:
                if p in history: errors.append(f"{m}/{s}: yasak ürün dönüşü {p}.")
                history.append(p)
            if (m,p) not in blocks:
                credit = initial["kg"] if previous==p and p==initial["product"] and not used[m] else 0
                blocks[m,p] = {"kg":0,"credit":credit,"first":t,"last":t}
            blocks[m,p]["kg"] += kg; blocks[m,p]["last"] = t
            totals[p] += kg; daily[m,p,s[0]] += kg; used[m].add(p)
            previous,prepared = p,1
        for a,b in d.sequences:
            order = ([initial["product"]] if initial["product"] else []) + [p for p in history if p!=initial["product"]]
            if a in order and b in order and order.index(a)>order.index(b):
                errors.append(f"{m}: sıra kuralı ihlali {a} -> {b}.")
    for (m,p,day),qty in daily.items():
        if qty>d.machine_product[m,p]["daily_kg"]: errors.append(f"{m}/{p}/Day-{day}: günlük tonaj aşıldı.")
    for (m,day),count in changes.items():
        if count>d.max_changes: errors.append(f"{m}/Day-{day}: {count} değişim, izin {d.max_changes}.")
    for p,r in d.targets.items():
        qty = totals[p]
        if qty>r["kg"]: errors.append(f"{p}: hedef üstü üretim.")
        if qty<r["min_kg"]: errors.append(f"{p}: zorunlu minimum sağlanmadı.")
        if r["mode"]=="HARD" and qty!=r["kg"]: errors.append(f"{p}: HARD hedef tamamlanmadı.")
        small = [(m,b) for (m,pp),b in blocks.items() if pp==p and b["kg"]+b["credit"]<d.machine_product[m,p]["min_kg"]]
        for m,b in small:
            if not d.settings["AllowFinalRunBelowMinimum"] or qty!=r["kg"] or len(small)>1:
                errors.append(f"{m}/{p}: minimum run ihlali; hedefi tamamlayan tek son-run istisnası gerekli.")
            elif any(bb["last"]>b["last"] for (mm,pp),bb in blocks.items() if pp==p and mm!=m):
                errors.append(f"{m}/{p}: minimum altındaki run en son biten run değil.")
    for scope,rule in d.allocation.items():
        if not(rule["hard"] or not rule["allow_split"]): continue
        groups = defaultdict(set)
        for (m,p),b in blocks.items():
            group = p if scope=="ALL_PRODUCTS" else family_of[p]
            groups[group].add(m)
        for group,ms in groups.items():
            if len(ms)>rule["max"]: errors.append(f"allocation_rules.{scope}.{group}: makine sınırı aşıldı.")
    return errors


def optimize(d):
    started = time.perf_counter()
    issues = preflight(d)
    if issues:
        return {"status":"INFEASIBLE","records":[],"events":[],"issues":issues,"validation":"NOT_RUN","runtime_sec":time.perf_counter()-started,"all_objectives_optimal":False}
    model,variables,stages = build_model(d)
    invalid = model.Validate()
    if invalid:
        return {"status":"MODEL_INVALID","records":[],"events":[],"issues":[invalid],"validation":"NOT_RUN","runtime_sec":time.perf_counter()-started,"all_objectives_optimal":False}
    budget_started = time.perf_counter()
    previous_solver = None
    events = []
    for index,stage in enumerate(stages):
        remaining = d.settings["TotalTimeLimitSec"]-(time.perf_counter()-budget_started)
        if remaining<=0.001:
            events.append({"stage":stage["name"],"status":"NOT_RUN_TOTAL_TIME_LIMIT","value":None,"bound":None,"unit":stage["unit"],"runtime_sec":0,"gap":None})
            break
        model.Maximize(stage["expr"]) if stage["direction"]=="MAX" else model.Minimize(stage["expr"])
        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = min(stage["seconds"],remaining)
        solver.parameters.num_search_workers = d.settings["NumSearchWorkers"]
        solver.parameters.random_seed = (d.settings["RandomSeed"]+index)%2147483647
        solver.parameters.log_search_progress = bool(d.settings["LogSearchProgress"])
        print(f"CP-SAT: {stage['name']} | süre sınırı {min(stage['seconds'],remaining):.1f} sn",flush=True)
        tick = time.perf_counter()
        status = solver.Solve(model)
        elapsed = time.perf_counter()-tick
        good = status in (cp_model.OPTIMAL,cp_model.FEASIBLE)
        value = int(solver.Value(stage["expr"])) if good else None
        bound = solver.BestObjectiveBound() if good else None
        events.append({"stage":stage["name"],"status":solver.StatusName(status),
                       "value":None if value is None else value*stage["factor"],
                       "bound":None if bound is None else bound*stage["factor"],"unit":stage["unit"],
                       "runtime_sec":elapsed,"gap":None if value is None else abs(value-bound)*stage["factor"],
                       "time_limit_sec":min(stage["seconds"],remaining)})
        print(f"  {solver.StatusName(status)} | değer {events[-1]['value']} {stage['unit']} | {elapsed:.2f} sn",flush=True)
        if not good:
            if previous_solver is None:
                reason = "Kısıtlar birlikte sağlanamıyor." if status==cp_model.INFEASIBLE else "Süre içinde geçerli plan bulunamadı; çözümsüzlük kanıtlanmış değildir."
                return {"status":solver.StatusName(status),"records":[],"events":events,"issues":[reason],"validation":"NOT_RUN","runtime_sec":time.perf_counter()-started,"all_objectives_optimal":False}
            d.warnings.append(f"{stage['name']}: {solver.StatusName(status)}; son doğrulanabilir çözüm korundu.")
            break
        previous_solver = solver
        if status!=cp_model.OPTIMAL and d.settings["StageCompletionPolicy"]=="REQUIRE_OPTIMAL":
            d.warnings.append(f"{stage['name']}: optimum kanıtlanmadı; REQUIRE_OPTIMAL nedeniyle alt amaçlar çalıştırılmadı.")
            break
        # The policy is explicit in Excel; a FEASIBLE incumbent is never labelled
        # an optimal lexicographic solution. Its bound/gap stays in the log.
        model.Add(stage["expr"]==value)
        model.ClearHints()
        for variable_index in range(len(model.Proto().variables)):
            variable = model.GetIntVarFromProtoIndex(variable_index)
            model.AddHint(variable,solver.Value(variable))
    if previous_solver is None:
        return {"status":"UNKNOWN","records":[],"events":events,"issues":["Çözüm için süre kalmadı."],"validation":"NOT_RUN","runtime_sec":time.perf_counter()-started,"all_objectives_optimal":False}
    records = extract_plan(d,previous_solver,variables)
    validation = validate_plan(d,records)
    optimal = len(events)==len(stages) and all(r["status"]=="OPTIMAL" for r in events)
    return {"status":"VALIDATION_FAILED" if validation else ("OPTIMAL" if optimal else "FEASIBLE"),
            "records":[] if validation else records,"events":events,"issues":validation,
            "validation":"FAIL" if validation else "PASS","runtime_sec":time.perf_counter()-started,
            "all_objectives_optimal":optimal and not validation}


def plan_summaries(d,result):
    records = result["records"]
    valid = result["validation"]=="PASS"
    run_ids = {}
    counter = 0
    for machine in d.machines:
        m = machine["Machine_ID"]
        assigned = sorted({r["p"] for r in records if r["m"]==m and r["p"]},
                          key=lambda p:min(r["t"] for r in records if r["m"]==m and r["p"]==p))
        for p in assigned:
            counter+=1; run_ids[m,p]=f"R{counter:03d}"
    product_rows = []
    for product in d.products:
        p = product["Product_ID"]
        target = d.targets[p]
        rr = [r for r in records if r["p"]==p]
        planned = sum(r["kg"] for r in rr)
        machines = [r["Machine"] for r in d.machines if any(a["m"]==r["Machine_ID"] for a in rr)]
        status = "NOT_PLANNED" if not valid else ("MET" if planned==target["kg"] else "SOFT_UNMET")
        product_rows.append([p,product["Product"],product["Family"],target["mode"],target["kg"]/1000,
                             planned/1000 if valid else None,(target["kg"]-planned)/1000 if valid else None,
                             0 if valid else None,(planned/target["kg"] if target["kg"] else 1.0) if valid else None,
                             ", ".join(machines),len(machines) if valid else None,len(machines) if valid else None,
                             status,target["pct"],target["min_kg"]/1000,
                             ("PASS" if planned>=target["min_kg"] else "FAIL") if valid else "NOT_CHECKED",
                             target["priority"]])
    machine_rows = []
    for machine in d.machines:
        m = machine["Machine_ID"]
        rr = [r for r in records if r["m"]==m and r["p"]]
        planned = sum(r["kg"] for r in rr)/1000
        production_h = sum(r["kg"]/1000/d.machine_product[m,r["p"]]["rate"] for r in rr)
        preparation_h = sum(r["Prep_min"] or 0 for r in rr)/60
        available_h = sum(d.hours[m,s] for s in d.slots)
        daily = defaultdict(int)
        for r in rr: daily[d.slots[r["t"]][0]]+=r["Changeover"]
        machine_rows.append([m,machine["Machine"],planned if valid else None,available_h,
                             production_h if valid else None,preparation_h if valid else None,
                             production_h+preparation_h if valid else None,
                             max(0,available_h-production_h-preparation_h) if valid else None,
                             ((production_h+preparation_h)/available_h if available_h else 0) if valid else None,
                             len({r["p"] for r in rr}) if valid else None,len(rr and {r["p"] for r in rr} or []) if valid else None,
                             sum(daily.values()) if valid else None,max(daily.values(),default=0) if valid else None])
    return run_ids,product_rows,machine_rows


def write_output_excel(d,result,output_path):
    """Preserve the five established output views, including a validation log."""
    workbook = Workbook()
    workbook.remove(workbook.active)
    navy,light = "17365D","EAF2F8"
    def header(ws,row,width):
        for col in range(1,width+1):
            cell=ws.cell(row,col)
            cell.font=Font(name="Arial",size=10,bold=True,color="FFFFFF")
            cell.fill=PatternFill("solid",fgColor=navy)
            cell.alignment=Alignment(horizontal="center",vertical="center",wrap_text=True)
        ws.row_dimensions[row].height=30
    def fit(ws):
        for col in range(1,ws.max_column+1):
            length=max((len(str(ws.cell(row,col).value)) for row in range(1,ws.max_row+1) if ws.cell(row,col).value is not None),default=8)
            ws.column_dimensions[get_column_letter(col)].width=min(45,max(12,length+2))
        ws.sheet_view.showGridLines=False
        ws.page_setup.orientation="landscape"
        for row in ws.iter_rows():
            for cell in row:
                if not cell.font.bold: cell.font=Font(name="Arial",size=10)
                cell.alignment=Alignment(vertical="center",wrap_text=True)
    valid=result["validation"]=="PASS"
    run_ids,product_rows,machine_rows=plan_summaries(d,result) if d else ({},[],[])
    target=sum(r["kg"] for r in d.targets.values())/1000 if d else None
    planned=sum(r["kg"] for r in result["records"])/1000 if valid else None
    dashboard=workbook.create_sheet("dashboard")
    dashboard.append(["Metric","Value"])
    for row in [
        ["Total_Target_t",target],["Planned_t",planned],
        ["Unmet_t",round(target-planned,9) if valid else None],
        ["Fulfillment",(planned/target if target else 1) if valid else None],
        ["Runs",len(run_ids) if valid else None],
        ["Changeovers",sum(r["Changeover"] for r in result["records"]) if valid else None],
        ["Product_Minimums","ALL_PASS" if valid else "NOT_CHECKED"],
        ["Solver_Status",result["status"]],["Validation_Status",result["validation"]],
        ["All_Objectives_Optimal",result.get("all_objectives_optimal",False)],
        ["CP_SAT_Runtime_sec",result["runtime_sec"]],
        ["Code_Version",VERSION],["OR_Tools_Version",ortools.__version__],
        ["Input_File",result.get("input_name","")],["Input_SHA256",result.get("input_sha256","")],
        ["Run_UTC",result.get("run_utc","")],
    ]: dashboard.append(row)
    for r in (2,3,4,13): dashboard.cell(r,2).number_format="#,##0.000"
    dashboard.cell(5,2).number_format="0.000%"
    dashboard.column_dimensions["A"].width=30
    dashboard.column_dimensions["B"].width=78
    header(dashboard,1,2)
    plan=workbook.create_sheet("results-plan")
    plan.cell(1,1,"Day");plan.cell(1,2,"Shift")
    plan.merge_cells("A1:A2");plan.merge_cells("B1:B2")
    slot_map={(r["m"],r["t"]):r for r in result["records"]}
    if d:
        pnames={p["Product_ID"]:p["Product"] for p in d.products}
        for index,machine in enumerate(d.machines):
            m=machine["Machine_ID"];col=3+4*index
            plan.merge_cells(start_row=1,start_column=col,end_row=1,end_column=col+3)
            plan.cell(1,col,machine["Machine"])
            for j,label in enumerate(("Prep_min","Product","Planned_t","Run_ID")):plan.cell(2,col+j,label)
            for t,s in enumerate(d.slots):
                row=3+t;plan.cell(row,1,f"Day-{s[0]}");plan.cell(row,2,s[1])
                r=slot_map.get((m,t))
                if r and r["p"]:
                    values=[r["Prep_min"],pnames[r["p"]],r["kg"]/1000,run_ids[m,r["p"]]]
                else:
                    state="CLOSED" if d.hours[m,s]==0 else ("IDLE" if valid else "NO_SOLUTION")
                    values=[None,state,None,None]
                for j,value in enumerate(values):plan.cell(row,col+j,value)
                plan.cell(row,col+2).number_format="#,##0.000"
        header(plan,1,2+4*len(d.machines));header(plan,2,2+4*len(d.machines))
    plan.freeze_panes="C3"
    ps=workbook.create_sheet("product-summary")
    ps.append(["Product_ID","Product","Family","TargetMode","Target_t","Planned_t","Unmet_t","Surplus_t",
               "Fulfillment","Machines","Machine_Count","Runs","Target_Status","MinRequired_pct","MinRequired_t","Minimum_Status","TargetPriority"])
    for row in product_rows:ps.append(row)
    ps.append([None,"TOTAL",None,None,target,planned,round(target-planned,9) if valid else None,0 if valid else None])
    header(ps,1,17);ps.freeze_panes="C2";ps.auto_filter.ref=f"A1:Q{max(1,ps.max_row-1)}"
    for row in range(2,ps.max_row+1):
        for col in (5,6,7,8,15):ps.cell(row,col).number_format="#,##0.000"
        for col in (9,14):ps.cell(row,col).number_format="0.000%"
    ms=workbook.create_sheet("machine-summary")
    ms.append(["Machine_ID","Machine","Planned_t","Available_h","Production_h","Preparation_h","Occupied_h","Idle_h",
               "Utilization","Products","Runs","Changeovers","Max_Daily_Changes"])
    for row in machine_rows:ms.append(row)
    header(ms,1,13);ms.freeze_panes="C2"
    for row in range(2,ms.max_row+1):
        for col in range(3,9):ms.cell(row,col).number_format="#,##0.000"
        ms.cell(row,9).number_format="0.0%"
    log=workbook.create_sheet("solver-log")
    log.append(["Stage","Status","Value","Best_Bound","Unit","Runtime_sec","Absolute_Gap","Time_Limit_sec"])
    for event in result["events"]:
        log.append([event.get(k) for k in ("stage","status","value","bound","unit","runtime_sec","gap","time_limit_sec")])
    header(log,1,8)
    log.append([]);log.append(["Independent validation",result["validation"]])
    if valid:
        log.append(["Checked rules","Calendar, eligibility, quantity step, shift capacity including preparation, daily tonnage, initial state, no return, sequence, daily changes, hard targets, minimum %, min run/final exception, allocation limits"])
    for issue in result["issues"]:log.append(["ERROR",issue])
    for warning in (d.warnings if d else []):log.append(["WARNING",warning])
    for ws in workbook.worksheets:fit(ws)
    dashboard.column_dimensions["B"].width=78
    log.column_dimensions["A"].width=42;log.column_dimensions["B"].width=95
    if d:
        for index in range(len(d.machines)):
            plan.column_dimensions[get_column_letter(4+4*index)].width=29
    output_path=Path(output_path)
    output_path.parent.mkdir(parents=True,exist_ok=True)
    workbook.save(output_path)
    return output_path


def run_pipeline(input_path,output_path=OUTPUT_XLSX):
    input_path=Path(input_path)
    started=time.perf_counter()
    d=None
    try:
        d=load_data(input_path)
        result=optimize(d)
    except InputValidationError as exc:
        result={"status":"INPUT_ERROR","records":[],"events":[],"issues":[str(exc)],"validation":"NOT_RUN",
                "runtime_sec":time.perf_counter()-started,"all_objectives_optimal":False}
    result["input_name"]=input_path.name
    result["input_sha256"]=hashlib.sha256(input_path.read_bytes()).hexdigest() if input_path.is_file() else ""
    result["run_utc"]=datetime.now(timezone.utc).isoformat(timespec="seconds")
    write_output_excel(d,result,output_path)
    result["total_wall_sec"]=time.perf_counter()-started
    print(f"Durum: {result['status']} | Bağımsız kontrol: {result['validation']}")
    if result["validation"]=="PASS":
        total=sum(v["kg"] for v in d.targets.values())
        planned=sum(r["kg"] for r in result["records"])
        print(f"Hedef {total/1000:.3f} t | Plan {planned/1000:.3f} t | Eksik {(total-planned)/1000:.3f} t")
    for issue in result["issues"]:print("HATA:",issue)
    for warning in (d.warnings if d else []):print("UYARI:",warning)
    print(f"Excel: {Path(output_path).resolve()}")
    return result


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument("input",nargs="?",type=Path,help="Yerel giriş Excel. Boşsa GitHub'dan her çalıştırmada indirilir.")
    parser.add_argument("--input-url",default=INPUT_GITHUB_URL)
    parser.add_argument("-o","--output",type=Path,default=OUTPUT_XLSX)
    args=parser.parse_args()
    input_path=args.input if args.input is not None else download_input_excel(args.input_url,INPUT_XLSX)
    result=run_pipeline(input_path,args.output)
    return 0 if result["validation"]=="PASS" else 2


## Giriş ve çıkış yolları
GitHub `blob` veya `raw.githubusercontent.com` bağlantısı kullanılabilir. İş kuralları ve çözücü ayarları giriş Excel'indedir.


In [ ]:
from pathlib import Path

INPUT_GITHUB_URL = (
    "https://github.com/hadokan/Production-planning/blob/main/"
    "CP_SAT_input_data_v1.xlsx"
)

WORK_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
INPUT_XLSX = WORK_DIR / "CP_SAT_input_data_v1.xlsx"
OUTPUT_XLSX = WORK_DIR / "CP_SAT_output.xlsx"


## İndir, çöz ve Excel çıktısını oluştur
Girdi hatası veya çözümsüzlük halinde de açıklamalı bir sonuç Excel'i oluşturulur; geçersiz plan başarı olarak gösterilmez. `FEASIBLE`, optimumun kanıtlanmadığı anlamına gelir.


In [ ]:
input_path = download_input_excel(INPUT_GITHUB_URL, INPUT_XLSX)
result = run_pipeline(input_path, OUTPUT_XLSX)

if result["validation"] == "PASS":
    print("Planın bağımsız kural kontrolü başarılı.")
else:
    print("Üretim planı oluşmadı. dashboard ve solver-log sayfalarındaki açıklamaları kontrol edin.")


## Sonuç Excel'ini indir


In [ ]:
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(OUTPUT_XLSX)))
else:
    files.download(str(OUTPUT_XLSX))
